# 🍕 Linear Regression Deep Dive with Pizza Store Data

**Complete Linear Regression Course** covering:
- Simple Linear Regression (by hand + sklearn)
- Multiple Linear Regression
- Normal Equation vs Gradient Descent
- Assumptions & Diagnostics
- Evaluation Metrics (MSE, RMSE, R², Adjusted R²)
- Regularization (Ridge, Lasso, Elastic Net)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

### 🤔 Understanding the dataset

We create 200 pizza stores with a **known true relationship**:

`daily_sales = 30 + 25×employees + 50×rating + 1.5×ad_spend + noise`

The `noise` simulates real-world randomness — even with the same employees, rating, and ad spend, two stores won't have identical sales. By knowing the true formula, we can check if our model recovers the correct coefficients.

In [ ]:
# Create Pizza Store Dataset
n_stores = 200
np.random.seed(42)

employees = np.random.randint(2, 12, n_stores)
rating = np.round(np.random.uniform(2.5, 5.0, n_stores), 1)
ad_spend = np.random.randint(50, 300, n_stores)

# True relationship: sales = 30 + 25*employees + 50*rating + 1.5*ad_spend + noise
daily_sales = (30 + 25*employees + 50*rating + 1.5*ad_spend
               + np.random.normal(0, 30, n_stores)).astype(int)

df = pd.DataFrame({
    'employees': employees,
    'rating': rating,
    'ad_spend': ad_spend,
    'daily_sales': daily_sales
})

print(f'Dataset: {len(df)} pizza stores')
print(f'Sales range: ${df.daily_sales.min()} — ${df.daily_sales.max()}')
df.head(8)

---
## Part 1: Simple Linear Regression — By Hand

### 1.1 The Math: β₁ = Cov(x,y) / Var(x), β₀ = ȳ − β₁x̄

### 🤔 What are Covariance and Variance, and why do we need them?

To find the best-fit line, we need two statistics:

---

**Covariance (Cov)** measures how two variables move **together**:
- Positive Cov: when x goes up, y tends to go up (more employees → more sales)
- Negative Cov: when x goes up, y tends to go down
- Zero Cov: no relationship

**Formula:**
```
Cov(x, y) = (1/n) × Σ (xᵢ − x̄)(yᵢ − ȳ)
```

**How to read this:** For each data point, ask: "How far is this x from the average x?" and "How far is this y from the average y?" Multiply those two distances together. Then average across all data points.

**Step-by-step example** (with 4 stores):
```
Store  Employees(x)  Sales(y)   (xᵢ − x̄)   (yᵢ − ȳ)   Product
─────  ────────────  ────────   ─────────   ─────────   ───────
  S1        5          520       5−3.5=1.5   520−400=120   1.5 × 120 = 180
  S2        2          280       2−3.5=−1.5  280−400=−120  −1.5 × −120 = 180
  S3        4          450       4−3.5=0.5   450−400=50    0.5 × 50 = 25
  S4        3          350       3−3.5=−0.5  350−400=−50   −0.5 × −50 = 25

x̄ = (5+2+4+3)/4 = 3.5       ȳ = (520+280+450+350)/4 = 400

Cov(x,y) = (180 + 180 + 25 + 25) / 4 = 410 / 4 = 102.5
```

**Key insight:** When x is above average AND y is above average, the product is **positive**. When both are below average, the product is also **positive** (negative × negative). So if x and y move together, the covariance is positive.

---

**Variance (Var)** measures how spread out a single variable is:
- High Var: values are far from the mean
- Low Var: values are clustered near the mean

**Formula:**
```
Var(x) = (1/n) × Σ (xᵢ − x̄)²
```

**How to read this:** For each data point, ask: "How far is this x from the average?" Square that distance (to make it positive). Then average across all data points.

**Step-by-step example** (same 4 stores):
```
Store  Employees(x)   (xᵢ − x̄)   (xᵢ − x̄)²
─────  ────────────   ─────────   ──────────
  S1        5          1.5         2.25
  S2        2         −1.5         2.25
  S3        4          0.5         0.25
  S4        3         −0.5         0.25

Var(x) = (2.25 + 2.25 + 0.25 + 0.25) / 4 = 5.0 / 4 = 1.25
```

Notice: Variance is just Covariance of x with **itself** — Var(x) = Cov(x, x).

---

**Why β₁ = Cov(x,y) / Var(x)?**

```
β₁ = Cov(x,y) / Var(x) = 102.5 / 1.25 = 82.0
```

The slope tells us: "for each unit increase in x, how much does y change?" Covariance captures the joint movement, and dividing by Var(x) **normalizes** it to a per-unit-of-x scale.

Think of it this way:
- Cov(x,y) tells you the **raw** relationship strength
- Var(x) tells you how much x varies
- Dividing gives you the relationship **per unit of x** — which is exactly what a slope is!

This is the **Ordinary Least Squares (OLS)** formula — it gives the line that minimizes the sum of squared errors.

In [ ]:
x = df['employees'].values
y = df['daily_sales'].values

# Step-by-step calculation
x_mean = x.mean()
y_mean = y.mean()
cov_xy = np.mean((x - x_mean) * (y - y_mean))
var_x = np.mean((x - x_mean) ** 2)

beta_1 = cov_xy / var_x
beta_0 = y_mean - beta_1 * x_mean

print('SIMPLE LINEAR REGRESSION — BY HAND')
print('=' * 50)
print(f'x̄ (mean employees) = {x_mean:.2f}')
print(f'ȳ (mean sales)     = ${y_mean:.2f}')
print(f'Cov(x, y)          = {cov_xy:.2f}')
print(f'Var(x)             = {var_x:.2f}')
print(f'\nβ₁ = Cov/Var = {cov_xy:.2f} / {var_x:.2f} = {beta_1:.2f}')
print(f'β₀ = ȳ − β₁x̄ = {y_mean:.2f} − {beta_1:.2f}×{x_mean:.2f} = {beta_0:.2f}')
print(f'\n📐 Model: ŷ = {beta_0:.2f} + {beta_1:.2f} × Employees')
print(f'\nInterpretation:')
print(f'  Each additional employee → +${beta_1:.2f} daily sales')
print(f'  Baseline (0 employees)  → ${beta_0:.2f} (theoretical)')

### 1.2 Visualize the Best-Fit Line

### 🤔 Why visualize the regression line?

A plot of the data with the best-fit line helps you:
- **Verify** the model makes sense (does the line follow the data?)
- **Spot problems** like outliers pulling the line, or a curved pattern that a straight line misses
- **Communicate** results to non-technical stakeholders

The green dashed lines are **residuals** — the vertical distance from each point to the line. The model minimizes the sum of these squared distances.

In [ ]:
y_pred_simple = beta_0 + beta_1 * x

plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.5, color='steelblue', label='Actual stores')
plt.plot(sorted(x), [beta_0 + beta_1*xi for xi in sorted(x)],
         'r-', linewidth=2, label=f'ŷ = {beta_0:.1f} + {beta_1:.1f}x')

# Draw residuals for a few points
for i in range(0, 15, 3):
    plt.plot([x[i], x[i]], [y[i], y_pred_simple[i]],
             'g--', alpha=0.5, linewidth=1)

plt.xlabel('Employees')
plt.ylabel('Daily Sales ($)')
plt.title('Simple Linear Regression: Sales vs Employees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 1.3 Verify with sklearn

In [ ]:
model_simple = LinearRegression()
model_simple.fit(x.reshape(-1, 1), y)

print('sklearn vs Hand Calculation:')
print(f'  β₀: sklearn={model_simple.intercept_:.2f}, hand={beta_0:.2f}')
print(f'  β₁: sklearn={model_simple.coef_[0]:.2f}, hand={beta_1:.2f}')
print('\n✅ They match!')

---
## Part 2: Evaluation Metrics

### 🤔 Why do we need evaluation metrics?

A model is useless if we can't measure how good it is. Key metrics:

- **MSE** (Mean Squared Error): Average of squared errors. Penalizes large errors heavily.
- **RMSE** (Root MSE): Square root of MSE — same units as y (dollars). Easier to interpret.
- **MAE** (Mean Absolute Error): Average of absolute errors. Less sensitive to outliers.
- **R²** (R-squared): Proportion of variance explained. 0 = terrible, 1 = perfect.
- **Adjusted R²**: R² penalized for extra features — prevents gaming R² by adding useless features.

---

### �� Why Mean SQUARED Error? Why not just Mean Error?

This is a great question. Let's see what happens with plain Mean Error:

```
Store   Actual   Predicted   Error (actual − predicted)
─────   ──────   ─────────   ─────────────────────────
  S1     $520      $530        520 − 530 = −10  (overpredicted)
  S2     $310      $290        310 − 290 = +20  (underpredicted)
  S3     $450      $460        450 − 460 = −10  (overpredicted)

Mean Error = (−10 + 20 + −10) / 3 = 0 / 3 = 0  ← LOOKS PERFECT!
```

**The problem:** Positive and negative errors **cancel each other out**. A Mean Error of 0 doesn't mean the model is perfect — it just means the overestimates and underestimates happen to balance. The model is still off by $10–$20 on every prediction!

**Three ways to fix this:**

```
Option 1: Mean Absolute Error (MAE) — take |absolute value| of each error
  MAE = (|−10| + |20| + |−10|) / 3 = (10 + 20 + 10) / 3 = 13.33

Option 2: Mean Squared Error (MSE) — square each error
  MSE = ((-10)² + 20² + (-10)²) / 3 = (100 + 400 + 100) / 3 = 200

Option 3: RMSE — take √MSE to get back to dollar units
  RMSE = √200 = $14.14
```

**Both MAE and MSE solve the cancellation problem. So why prefer MSE?**

| Property | MAE | MSE |
|----------|-----|-----|
| Cancellation problem | ✅ Solved | ✅ Solved |
| Penalizes large errors | Equally (linear) | **More heavily** (quadratic) |
| Differentiable everywhere | ❌ Has a kink at 0 | ✅ Smooth everywhere |
| Optimization | Harder | **Easier** (gradient descent works better) |
| Sensitive to outliers | Less | More |

**The key reasons MSE is the default:**

1. **Squaring punishes big errors much more.** An error of 10 costs 100, but an error of 20 costs 400 (4× more, not 2×). This is often what we want — a model that's off by $100 once is worse than one that's off by $10 ten times.

2. **MSE is smooth (differentiable everywhere).** MAE has a sharp corner (kink) at error = 0, which makes gradient descent struggle. MSE's smooth parabolic shape gives clean gradients that point straight to the minimum.

```
  Loss                          Loss
   |\      /|  MAE               |         MSE
   | \    / |  (V-shape,         |  \     /
   |  \  /  |   kink at 0)       |   \   /
   |   \/   |                    |    \_/   (smooth bowl,
   +--------→ error              +--------→  easy to optimize)
```

3. **MSE has a closed-form solution.** Because of the smooth math, we can derive the Normal Equation (β = (XᵀX)⁻¹Xᵀy) directly. With MAE, there's no closed-form — you can only use iterative methods.

**When to use MAE instead:** When you have outliers and don't want them to dominate the loss. For example, if one store has a data entry error ($50,000 instead of $500), MSE would be dominated by that single point, while MAE would handle it more gracefully.

**Bottom line:** MSE is the default because it's mathematically convenient and penalizes big errors. Use MAE when outliers are a concern.

### 🤔 What is R² and why do we need it?

MSE and RMSE tell you the **size** of the errors, but they don't tell you if that's good or bad. Is an RMSE of $15 good? It depends — $15 is great if sales range from $100 to $1000, but terrible if sales range from $10 to $20.

**R² (R-squared)** solves this by measuring the **proportion of variance explained** by the model. It answers: "How much better is my model than just predicting the average every time?"

---

**The intuition:**

Imagine the dumbest possible model — it ignores all features and always predicts the **mean** of y (e.g., "every store makes $405"). This "dumb model" has some total error. R² measures how much of that error your model eliminates.

```
Dumb model (predict mean every time):
  S1: actual=$520, predict=$405, error² = (520−405)² = 13,225
  S2: actual=$310, predict=$405, error² = (310−405)² =  9,025
  S3: actual=$620, predict=$405, error² = (620−405)² = 46,225
  SS_tot = 13,225 + 9,025 + 46,225 + ... = total squared error of the dumb model

Your model (uses features):
  S1: actual=$520, predict=$529, error² = (520−529)² =     81
  S2: actual=$310, predict=$330, error² = (310−330)² =    400
  S3: actual=$620, predict=$628, error² = (620−628)² =     64
  SS_res = 81 + 400 + 64 + ... = total squared error of YOUR model
```

---

**The formula:**

```
R² = 1 − (SS_res / SS_tot)

Where:
  SS_tot = Σ(yᵢ − ȳ)²    ← total variance (how far each point is from the mean)
  SS_res = Σ(yᵢ − ŷᵢ)²   ← residual variance (how far each point is from your prediction)
```

**How to read it:**

```
R² = 0.0  → Your model is no better than predicting the mean (useless)
R² = 0.5  → Your model explains 50% of the variance (decent)
R² = 0.9  → Your model explains 90% of the variance (great)
R² = 1.0  → Your model explains 100% of the variance (perfect — suspicious!)
R² < 0    → Your model is WORSE than predicting the mean (something is very wrong)
```

**Visual way to think about it:**

```
Total Variance (SS_tot)
┌─────────────────────────────────────────────────┐
│                                                 │
│   Explained by model (SS_reg)     Unexplained   │
│   ████████████████████████████    ░░ (SS_res)   │
│                                                 │
└─────────────────────────────────────────────────┘

R² = Explained / Total = 1 − Unexplained / Total
```

---

**Why Adjusted R²?**

R² has a flaw: it **always increases** when you add more features, even useless ones. Adding "store_id" as a feature would increase R², but the model would be worse on new data.

**Adjusted R²** penalizes for extra features:

```
Adjusted R² = 1 − [(1 − R²)(n − 1) / (n − p − 1)]

Where:
  n = number of data points
  p = number of features
```

If a new feature doesn't improve the model enough to justify its complexity, Adjusted R² will **drop**. Rule: if Adjusted R² goes down when you add a feature, remove that feature.

In [ ]:
# Compute all metrics step by step
residuals = y - y_pred_simple
n = len(y)
p = 1  # one feature

mse = np.mean(residuals**2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(residuals))
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - y_mean)**2)
r2 = 1 - ss_res / ss_tot
adj_r2 = 1 - ((1 - r2) * (n - 1)) / (n - p - 1)

print('EVALUATION METRICS')
print('=' * 50)
print(f'MSE  = (1/n)Σ(yᵢ − ŷᵢ)²  = {mse:.2f}')
print(f'RMSE = √MSE               = ${rmse:.2f}')
print(f'MAE  = (1/n)Σ|yᵢ − ŷᵢ|    = ${mae:.2f}')
print(f'\nSS_res = Σ(yᵢ − ŷᵢ)²    = {ss_res:.2f}')
print(f'SS_tot = Σ(yᵢ − ȳ)²       = {ss_tot:.2f}')
print(f'R²     = 1 − SS_res/SS_tot = {r2:.4f}')
print(f'Adj R² = 1 − [(1−R²)(n−1)/(n−p−1)] = {adj_r2:.4f}')
print(f'\n📊 Employees alone explains {r2*100:.1f}% of sales variance')
print(f'   Predictions are off by ~${rmse:.0f} on average')

---
## Part 3: Multiple Linear Regression

### 🤔 Why use multiple features instead of just one?

Simple regression uses one feature. But daily sales depend on **many things** — employees, rating, AND ad spend. Multiple regression uses all of them simultaneously.

**The key insight:** Each coefficient now represents that feature's **unique contribution**, holding all other features constant. This is called **ceteris paribus** ("all else equal").

You might notice the Employees coefficient changes from simple to multiple regression. That's because in the simple model, Employees was a **proxy** for everything — stores with more employees also tend to have higher ratings and bigger ad budgets. Multiple regression separates these effects.

In [ ]:
# Multiple regression with all features
features = ['employees', 'rating', 'ad_spend']
X = df[features]
y = df['daily_sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)

print('MULTIPLE LINEAR REGRESSION')
print('=' * 50)
print(f'ŷ = {model_multi.intercept_:.2f}', end='')
for feat, coef in zip(features, model_multi.coef_):
    sign = '+' if coef >= 0 else ''
    print(f' {sign}{coef:.2f}×{feat}', end='')
print()

print(f'\nCoefficient Interpretation (holding others constant):')
for feat, coef in zip(features, model_multi.coef_):
    print(f'  {feat}: each +1 → ${coef:.2f} daily sales')

r2_multi = r2_score(y_test, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_multi))
print(f'\nR² = {r2_multi:.4f} (vs simple: {r2:.4f})')
print(f'RMSE = ${rmse_multi:.2f} (vs simple: ${rmse:.2f})')
print(f'\n🎯 Multiple regression is much better — uses all the information!')

### 3.1 Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simple model
y_pred_test_simple = model_simple.predict(X_test[['employees']])
axes[0].scatter(y_test, y_pred_test_simple, alpha=0.5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual Sales ($)')
axes[0].set_ylabel('Predicted Sales ($)')
axes[0].set_title(f'Simple Model (R²={r2_score(y_test, y_pred_test_simple):.3f})')

# Multiple model
axes[1].scatter(y_test, y_pred_multi, alpha=0.5, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel('Actual Sales ($)')
axes[1].set_ylabel('Predicted Sales ($)')
axes[1].set_title(f'Multiple Model (R²={r2_multi:.3f})')

plt.suptitle('Actual vs Predicted: Closer to red line = better', fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 4: Normal Equation vs Gradient Descent

### 🤔 What is the Normal Equation?

The Normal Equation is a **closed-form solution** — you plug in the data and get the exact optimal weights in one computation:

**β = (XᵀX)⁻¹Xᵀy**

This works by setting the derivative of the loss function to zero and solving algebraically. It's like finding the bottom of a bowl by calculus instead of rolling a ball downhill.

**Pros:** Exact answer, no hyperparameters, one computation.
**Cons:** Requires matrix inversion (O(p³) — slow for many features), fails if features are perfectly correlated (singular matrix).

In [ ]:
# Normal Equation: β = (XᵀX)⁻¹Xᵀy
X_mat = np.column_stack([np.ones(len(X_train)), X_train.values])
y_vec = y_train.values

beta_normal = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_vec

print('NORMAL EQUATION: β = (XᵀX)⁻¹Xᵀy')
print('=' * 50)
print(f'β₀ (intercept):  {beta_normal[0]:.4f}')
for i, feat in enumerate(features):
    print(f'β{i+1} ({feat:>10}): {beta_normal[i+1]:.4f}')

print(f'\nsklearn intercept: {model_multi.intercept_:.4f}')
print(f'sklearn coefs:     {model_multi.coef_}')
print('\n✅ Normal equation gives the same answer as sklearn!')

### 🤔 What is Gradient Descent and why do we need it?

Gradient descent is an **iterative** algorithm — it starts with random weights and improves them step by step:

1. Compute predictions with current weights
2. Compute the **gradient** (direction of steepest increase in error)
3. Move weights in the **opposite direction** (downhill)
4. Repeat until the error stops decreasing

**Why not always use the Normal Equation?** For large datasets (millions of rows, thousands of features), the matrix inversion is too slow. Gradient descent scales much better.

**Why scale features first?** Without scaling, features with large values (ad_spend: 50–300) dominate the gradient, causing zig-zagging and slow convergence. StandardScaler puts all features on the same scale (mean=0, std=1).

In [ ]:
# Gradient Descent from scratch
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add intercept column
X_gd = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
n_samples, n_features = X_gd.shape

# Initialize
beta = np.zeros(n_features)
lr = 0.01
n_iters = 1000
loss_history = []

for i in range(n_iters):
    y_hat = X_gd @ beta
    residuals = y_train.values - y_hat
    mse_loss = np.mean(residuals**2)
    loss_history.append(mse_loss)
    gradient = -(2/n_samples) * (X_gd.T @ residuals)
    beta = beta - lr * gradient

print('GRADIENT DESCENT FROM SCRATCH')
print('=' * 50)
print(f'Learning rate: {lr}, Iterations: {n_iters}')
print(f'Initial MSE: {loss_history[0]:.2f}')
print(f'Final MSE:   {loss_history[-1]:.2f}')
print(f'\nConverged weights (scaled features):')
print(f'  β₀={beta[0]:.2f}, β₁={beta[1]:.2f}, β₂={beta[2]:.2f}, β₃={beta[3]:.2f}')

# Plot convergence
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='steelblue')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Gradient Descent Convergence')
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 5: Assumptions & Diagnostics

### 🤔 Why check assumptions?

Linear regression makes assumptions about the data. If these are violated, the model's predictions and confidence intervals may be **unreliable**.

The 4 diagnostic plots below check:
1. **Residuals vs Predicted**: Should be random scatter (no pattern = linearity ✅, uniform band = homoscedasticity ✅)
2. **Q-Q Plot**: Points should follow the diagonal line (= normally distributed errors ✅)
3. **Histogram of Residuals**: Should be bell-shaped (= normal distribution ✅)
4. **Correlation Heatmap**: No feature pair should have correlation > 0.8 (= no multicollinearity ✅)

If you see a curved pattern in the residual plot, your model is missing a non-linear relationship. If the residuals fan out (cone shape), the variance isn't constant — try log-transforming y.

In [ ]:
residuals_multi = y_test.values - y_pred_multi

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Predicted (Linearity + Homoscedasticity)
axes[0,0].scatter(y_pred_multi, residuals_multi, alpha=0.5, color='steelblue')
axes[0,0].axhline(y=0, color='r', linestyle='--')
axes[0,0].set_xlabel('Predicted')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('1. Residuals vs Predicted\n(Check: random scatter = good)')

# 2. Q-Q Plot (Normality)
from scipy import stats
sorted_res = np.sort(residuals_multi)
theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_res)))
axes[0,1].scatter(theoretical, sorted_res, alpha=0.5, color='coral')
axes[0,1].plot([-3, 3], [-3*np.std(residuals_multi), 3*np.std(residuals_multi)], 'r--')
axes[0,1].set_xlabel('Theoretical Quantiles')
axes[0,1].set_ylabel('Sample Quantiles')
axes[0,1].set_title('2. Q-Q Plot\n(Check: points on line = normal)')

# 3. Histogram of Residuals
axes[1,0].hist(residuals_multi, bins=20, color='steelblue', edgecolor='white', alpha=0.7)
axes[1,0].set_xlabel('Residual')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title('3. Residual Distribution\n(Check: bell-shaped = normal)')

# 4. Feature Correlation Heatmap (Multicollinearity)
corr = df[features].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1,1],
            fmt='.2f', square=True)
axes[1,1].set_title('4. Feature Correlations\n(Check: no high correlations)')

plt.suptitle('Linear Regression Diagnostic Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Assumption Checks:')
print(f'  Residual mean: {np.mean(residuals_multi):.2f} (should be ~0) ✅')
print(f'  Residual std:  {np.std(residuals_multi):.2f}')
shapiro_stat, shapiro_p = stats.shapiro(residuals_multi)
print(f'  Shapiro-Wilk normality p-value: {shapiro_p:.4f}', '✅' if shapiro_p > 0.05 else '⚠️')

---
## Part 6: Regularization — Ridge, Lasso, Elastic Net

### 🤔 What is Regularization and why do we need it?

With many features, a model can **overfit** — it memorizes the training data (including noise) and performs poorly on new data. Regularization adds a **penalty** to the loss function that discourages large coefficients:

- **Ridge (L2)**: Penalty = λΣβⱼ² → shrinks all coefficients toward zero, but never to exactly zero
- **Lasso (L1)**: Penalty = λΣ|βⱼ| → can shrink coefficients to **exactly zero** (feature selection!)
- **Elastic Net**: Combines both L1 and L2

**λ (alpha)** controls the penalty strength:
- λ = 0: No penalty (standard OLS)
- λ = large: Heavy penalty (coefficients near zero, underfitting)

Use **cross-validation** to find the best λ.

In [ ]:
# Compare OLS vs Ridge vs Lasso vs Elastic Net
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

print('REGULARIZATION COMPARISON')
print('=' * 70)
print(f'{"Model":<25} {"RMSE":>8} {"R²":>8}  Coefficients')
print('-' * 70)

# OLS baseline
ols = LinearRegression().fit(X_train_scaled, y_train)
ols_pred = ols.predict(X_test_scaled)
print(f'{"OLS (no penalty)":<25} {np.sqrt(mean_squared_error(y_test, ols_pred)):>8.2f} {r2_score(y_test, ols_pred):>8.4f}  {np.round(ols.coef_, 2)}')

# Ridge (L2)
for alpha in [0.1, 1.0, 10.0]:
    ridge = Ridge(alpha=alpha).fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    print(f'{f"Ridge (α={alpha})":<25} {np.sqrt(mean_squared_error(y_test, ridge_pred)):>8.2f} {r2_score(y_test, ridge_pred):>8.4f}  {np.round(ridge.coef_, 2)}')

# Lasso (L1)
for alpha in [0.1, 1.0, 10.0]:
    lasso = Lasso(alpha=alpha).fit(X_train_scaled, y_train)
    lasso_pred = lasso.predict(X_test_scaled)
    print(f'{f"Lasso (α={alpha})":<25} {np.sqrt(mean_squared_error(y_test, lasso_pred)):>8.2f} {r2_score(y_test, lasso_pred):>8.4f}  {np.round(lasso.coef_, 2)}')

# Elastic Net
enet = ElasticNet(alpha=1.0, l1_ratio=0.5).fit(X_train_scaled, y_train)
enet_pred = enet.predict(X_test_scaled)
print(f'{"Elastic Net (α=1.0)":<25} {np.sqrt(mean_squared_error(y_test, enet_pred)):>8.2f} {r2_score(y_test, enet_pred):>8.4f}  {np.round(enet.coef_, 2)}')

print('\nKey observations:')
print('  • Ridge shrinks coefficients but never zeros them')
print('  • Lasso can zero out coefficients (feature selection!)')
print('  • High α → more shrinkage → simpler model')

### 6.1 Regularization Path — How Coefficients Shrink

In [ ]:
alphas_range = np.logspace(-2, 3, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge path
ridge_coefs = []
for a in alphas_range:
    ridge = Ridge(alpha=a).fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)
ridge_coefs = np.array(ridge_coefs)

for i, feat in enumerate(features):
    axes[0].plot(alphas_range, ridge_coefs[:, i], label=feat)
axes[0].set_xscale('log')
axes[0].set_xlabel('α (regularization strength)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Ridge (L2): Coefficients shrink but never reach 0')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Lasso path
lasso_coefs = []
for a in alphas_range:
    lasso = Lasso(alpha=a, max_iter=10000).fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)
lasso_coefs = np.array(lasso_coefs)

for i, feat in enumerate(features):
    axes[1].plot(alphas_range, lasso_coefs[:, i], label=feat)
axes[1].set_xscale('log')
axes[1].set_xlabel('α (regularization strength)')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso (L1): Coefficients can hit exactly 0')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Regularization Paths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 7: Polynomial Regression — When Linear Isn't Enough

### 🤔 What if the relationship isn't linear?

Linear regression fits a straight line. But what if the true relationship is curved? **Polynomial regression** adds powers of x (x², x³, ...) as new features:

- Degree 1: ŷ = β₀ + β₁x (straight line — may underfit)
- Degree 2: ŷ = β₀ + β₁x + β₂x² (parabola — often just right)
- Degree 15: ŷ = β₀ + β₁x + ... + β₁₅x¹⁵ (wiggly — overfits!)

This is a perfect demonstration of the **bias-variance tradeoff**:
- Too simple → misses the pattern (high bias)
- Too complex → fits the noise (high variance)
- Just right → captures the true pattern

In [ ]:
# Demonstrate underfitting vs overfitting with polynomial degree
np.random.seed(42)
x_poly = np.sort(np.random.uniform(0, 10, 30))
y_poly = 3 + 2*x_poly - 0.3*x_poly**2 + np.random.normal(0, 2, 30)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_plot = np.linspace(0, 10, 100)

for ax, degree, title in zip(axes, [1, 2, 15],
    ['Degree 1: Underfitting', 'Degree 2: Just Right', 'Degree 15: Overfitting']):
    poly = PolynomialFeatures(degree)
    X_p = poly.fit_transform(x_poly.reshape(-1, 1))
    model_p = LinearRegression().fit(X_p, y_poly)
    X_plot = poly.transform(x_plot.reshape(-1, 1))
    y_plot = model_p.predict(X_plot)

    train_r2 = model_p.score(X_p, y_poly)
    ax.scatter(x_poly, y_poly, alpha=0.6, color='steelblue')
    ax.plot(x_plot, y_plot, 'r-', linewidth=2)
    ax.set_title(f'{title}\nTrain R²={train_r2:.3f}')
    ax.set_ylim(y_poly.min()-5, y_poly.max()+5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff with Polynomial Degree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key takeaway:')
print('  Degree 1:  High bias (misses the curve)')
print('  Degree 2:  Just right (captures the true pattern)')
print('  Degree 15: High variance (fits noise, will fail on new data)')

---
## Part 8: Cross-Validation

### 🤔 What is Cross-Validation and why is it better than a single train/test split?

A single train/test split is **random** — you might get lucky or unlucky with which data ends up in each set. **K-fold cross-validation** gives a more reliable estimate:

1. Split data into K equal parts (folds)
2. Train on K-1 folds, test on the remaining fold
3. Repeat K times (each fold gets to be the test set once)
4. Average the K scores

This uses **all data for both training and testing** (just not at the same time). The standard deviation of the K scores tells you how **stable** the model is — low std = consistent performance, high std = sensitive to which data it sees.

In [ ]:
# 5-fold cross-validation
models = {
    'OLS': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Lasso (α=1)': Lasso(alpha=1.0),
    'Elastic Net': ElasticNet(alpha=1.0, l1_ratio=0.5),
}

print('5-FOLD CROSS-VALIDATION')
print('=' * 50)
print(f'{"Model":<20} {"Mean R²":>10} {"Std":>10}')
print('-' * 50)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    print(f'{name:<20} {scores.mean():>10.4f} {scores.std():>10.4f}')

print('\nCross-validation gives a more reliable estimate of model performance')
print('than a single train/test split.')

---
## Summary

You've learned:
- **Simple Linear Regression**: β₁ = Cov(x,y)/Var(x), computed by hand
- **Multiple Linear Regression**: Isolating each feature's unique effect
- **Normal Equation vs Gradient Descent**: Two ways to find optimal weights
- **Evaluation**: MSE, RMSE, MAE, R², Adjusted R²
- **Assumptions**: Linearity, normality, homoscedasticity, independence
- **Regularization**: Ridge (L2), Lasso (L1), Elastic Net
- **Polynomial Regression**: When linear isn't enough
- **Cross-Validation**: Reliable model evaluation